In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
# import geopandas as gpd
# from shapely.geometry import Point
from lisfloodutilities.cutmaps.cutlib import mask_from_ldd
from lisfloodutilities.nc2pcr import convert
from lisfloodutilities.readers import PCRasterMap

from tqdm import tqdm
import os
import shutil
import tempfile
import logging


In [21]:
# LDD direction convention (PCRaster):
# 7 8 9
# 4 5 6
# 1 2 3
# For each direction, what is the (row_offset, col_offset) of the downstream neighbour?
# e.g. direction 1 means flow goes to bottom-left → downstream is at (+1, -1)

# Inverse: for each neighbour position, which LDD value points TO the current cell?
# neighbour at (-1, -1) has direction 3 if it flows to us
# neighbour at (-1,  0) has direction 2
# neighbour at (-1, +1) has direction 1
# neighbour at ( 0, -1) has direction 6
# neighbour at ( 0, +1) has direction 4
# neighbour at (+1, -1) has direction 9
# neighbour at (+1,  0) has direction 8
# neighbour at (+1, +1) has direction 7

# (row_offset, col_offset) -> LDD value that points toward current cell
UPSTREAM_DIRS = {
    (-1, -1): 3,
    (-1,  0): 2,
    (-1, +1): 1,
    ( 0, -1): 6,
    ( 0, +1): 4,
    ( 1, -1): 9,
    ( 1,  0): 8,
    ( 1, +1): 7,
}

def cutmaps_own(ldd_array, lon_array, lat_array, station_lon, station_lat):
    """
    Trace upstream from station pixel through LDD to get catchment mask.
    Returns a 2D boolean array (same shape as ldd_array).
    """
    # find nearest pixel to station coordinates
    col = np.argmin(np.abs(lon_array - station_lon))
    row = np.argmin(np.abs(lat_array - station_lat))
    # print(f"  Station pixel: row={row}, col={col}, lon={lon_array[col]:.3f}, lat={lat_array[row]:.3f}")

    nrows, ncols = ldd_array.shape
    mask = np.zeros((nrows, ncols), dtype=bool)

    # BFS upstream flood-fill
    queue = [(row, col)]
    mask[row, col] = True

    while queue:
        r, c = queue.pop()
        for (dr, dc), upstream_val in UPSTREAM_DIRS.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < nrows and 0 <= nc < ncols:
                if not mask[nr, nc] and ldd_array[nr, nc] == upstream_val:
                    mask[nr, nc] = True
                    queue.append((nr, nc))

    return mask

# =============================================================================
# SEASONALITY HELPER
# =============================================================================

def seasonality_index(monthly_means: np.ndarray) -> float:
    """
    Walsh & Lawler (1981) seasonality index.
    monthly_means: array of 12 values (climatological monthly means)
    SI = (1/R) * sum(|xi - R/12|)  where R = annual sum
    Returns 0 (perfectly uniform) to ~1.83 (all rain in one month).
    """
    monthly_means = np.array(monthly_means, dtype=float)
    R = monthly_means.sum()
    if R == 0:
        return np.nan
    return float(np.sum(np.abs(monthly_means - R / 12)) / R)


def get_area_cut(area_array, area_lon, area_lat, bbox_lon, bbox_lat):
    ac_min = np.argmin(np.abs(area_lon - bbox_lon[0]))
    ac_max = np.argmin(np.abs(area_lon - bbox_lon[1]))
    ar_min = np.argmin(np.abs(area_lat - bbox_lat[0]))
    ar_max = np.argmin(np.abs(area_lat - bbox_lat[1]))
    return area_array[ar_min:ar_max+1, ac_min:ac_max+1]

In [16]:
# BASE_FILE = Path.cwd () / "GloFASv5_stations_metadata_calfunction_KGE_JSD_20March2026_final.csv"
BASE_FILE = Path.cwd () / "GloFASv5_catchments_comprehensive.csv"
# Folder with climate files
DIR_CLIM = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/long_term_runs/00_Climate/")
# Folder with static attribute files
DIR_STATIC = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/static_maps/GloFASv5_staticmaps_consolidated_March2026/GloFASv5_static_maps_reanalysis/")

# Base File
glofas5_base_info = pd.read_csv(BASE_FILE)


In [ ]:
# station list
cal_stations = glofas5_base_info[["LISFLOOD_X","LISFLOOD_Y", "ID"]].values
if not (DIR_STATIC / "cal_stations.txt").exists():
    with open(DIR_STATIC / "cal_stations.txt", "w") as f:
        for x, y, id_ in cal_stations:
            f.write(f"{x:.3f}  {y:.3f}\t{int(id_)}\n")

### Load ldd & uparea

In [ ]:
# Load LDD file
ldd_nc = DIR_STATIC / "ldd_repaired.nc"
ds_ldd = xr.open_dataset(ldd_nc)
print("\n=== ldd_repaired.nc ===")
print(ds_ldd)
print("\n=== Variables & dtypes ===")
for var in ds_ldd.data_vars:
    print(f"{var}: {ds_ldd[var].dtype}")
ds_ldd.close()

# Load upArea file
uparea_nc = DIR_STATIC / "upArea_repaired_correctedmetadata_3000.nc"
ds_uparea = xr.open_dataset(uparea_nc)
print("\n=== upArea_repaired_correctedmetadata_3000.nc ===")
print(ds_uparea)
print("\n=== Variables & dtypes ===")
for var in ds_uparea.data_vars:
    print(f"{var}: {ds_uparea[var].dtype}")
ds_uparea.close()


=== ldd_repaired.nc ===
<xarray.Dataset>
Dimensions:  (lon: 7200, lat: 3000)
Coordinates:
  * lon      (lon) float64 -180.0 -179.9 -179.9 -179.8 ... 179.9 179.9 180.0
  * lat      (lat) float64 89.97 89.92 89.88 89.82 ... -59.88 -59.92 -59.97
Data variables:
    crs      |S1 ...
    Band1    (lat, lon) float32 ...
Attributes:
    CDI:                        Climate Data Interface version 1.9.10 (https:...
    Conventions:                CF-1.5
    GDAL_AREA_OR_POINT:         Area
    GDAL_PCRASTER_VALUESCALE:   VS_LDD
    GDAL:                       GDAL 3.2.1, released 2020/12/29
    CDO:                        Climate Data Operators version 1.9.10 (https:...
    history:                    Fri Dec 10 11:49:28 2021: ncks -A -v lat,lon ...
    history_of_appended_files:  Fri Dec 10 11:49:28 2021: Appended file ldd_O...
    NCO:                        netCDF Operators version 4.9.7 (Homepage = ht...

=== Variables & dtypes ===
crs: |S1
Band1: float32

=== upArea_repaired_correctedmetad

### pythonic catchment masking

In [9]:
# ── load LDD once ─────────────────────────────────────────────────────────────
ds_ldd = xr.open_dataset(DIR_STATIC / "ldd_repaired.nc")
ldd_array = ds_ldd['Band1'].values
lon_array = ds_ldd.lon.values
lat_array = ds_ldd.lat.values

# ── test on first station ─────────────────────────────────────────────────────
x, y, station_id = cal_stations[0]
station_id = int(station_id)
print(f"Testing station {station_id} at ({x}, {y})")

import time
t0 = time.time()
catchment_mask = cutmaps_own(ldd_array, lon_array, lat_array, x, y)
print(f"Done in {time.time()-t0:.2f}s")
print(f"Catchment size: {catchment_mask.sum()} pixels")

Testing station 4 at (11.875, 52.025)
Done in 0.05s
Catchment size: 4747 pixels


In [ ]:
plt.pcolormesh(catchment_mask)

### loop over all catchments

In [23]:
"""
catchment_meteo_attributes.py
------------------------------
Extracts catchment-average climate statistics for all GloFAS calibration
stations from yearly global NetCDF files (tp, eT0, ta).

Strategy:
  1. Compute BFS catchment masks for all stations once (on LDD)
  2. Loop over years x variables — load one file at a time
  3. For each station: slice bbox, apply mask, compute stats
  4. Aggregate across years -> long-term means, interannual std, seasonality, aridity index
"""

import time
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from tqdm import tqdm

# =============================================================================
# CONFIG
# =============================================================================

STATIC_DIR  = DIR_STATIC
CLIMATE_DIR = DIR_CLIM  # root dir where glofas_eT0_1999.nc etc. live

LDD_NC     = STATIC_DIR / "ldd_repaired.nc"
PIXAREA_NC = STATIC_DIR / "pixarea_Global_03min.nc"

CLIM_VARS  = ["tp", "eT0", "ta"]
YEARS      = range(1991, 2021)  # adjust to your available years

def get_clim_path(var: str, year: int) -> Path:
    return CLIMATE_DIR / f"glofas_{var}_{year}.nc"


# =============================================================================
# STEP 0 — load LDD + pixel area, compute all BFS masks
# =============================================================================

print("Loading LDD...")
ds_ldd    = xr.open_dataset(LDD_NC)
ldd_array = np.squeeze(ds_ldd['Band1'].values)
lon_array = ds_ldd.lon.values
lat_array = ds_ldd.lat.values
print(f"LDD shape: {ldd_array.shape}")

print("Loading pixel area...")
ds_area    = xr.open_dataset(PIXAREA_NC)
area_array = np.squeeze(ds_area[list(ds_area.data_vars)[-1]].values)
area_lon   = ds_area.lon.values
area_lat   = ds_area.lat.values

print("Computing BFS catchment masks for all stations...")
cal_stations = glofas5_base_info[["LISFLOOD_X", "LISFLOOD_Y", "ID"]].values

station_masks = {}
t0 = time.time()
for x, y, station_id in tqdm(cal_stations, desc="BFS masks"):
    station_id = int(station_id)
    mask = cutmaps_own(ldd_array, lon_array, lat_array, x, y)
    rows_idx, cols_idx = np.where(mask)
    if len(rows_idx) == 0:
        station_masks[station_id] = None
        continue
    r_min, r_max = rows_idx.min(), rows_idx.max()
    c_min, c_max = cols_idx.min(), cols_idx.max()
    mask_cut = mask[r_min:r_max+1, c_min:c_max+1]
    station_masks[station_id] = (mask_cut, r_min, r_max, c_min, c_max)

print(f"All masks done in {time.time()-t0:.1f}s")


# =============================================================================
# STEP 1 — loop variables x years, accumulate per station
# =============================================================================

results_accum = {
    int(sid): {
        var: {
            "annual":         [],
            "monthly_sums":   np.zeros(12),
            "monthly_counts": np.zeros(12),
        }
        for var in CLIM_VARS
    }
    for _, _, sid in cal_stations
}

for var in CLIM_VARS:
    for year in tqdm(YEARS, desc=f"{var}"):
        fpath = get_clim_path(var, year)
        if not fpath.exists():
            print(f"  WARNING: missing {fpath.name}, skipping")
            continue

        # open metadata only
        ds = xr.open_dataset(fpath)
        da = ds[var]
        clim_lon = da.lon.values
        clim_lat = da.lat.values
        months   = da.time.dt.month.values  # (n_timesteps,)

        # KEY FIX: load full array into memory ONCE per year
        print(f"  Loading {fpath.name} into memory...", end=" ", flush=True)
        t_load = time.time()
        arr = da.values.astype(float)  # (366, 3000, 7200)
        print(f"{arr.nbytes/1e9:.1f} GB in {time.time()-t_load:.1f}s")
        ds.close()

        # precompute climate grid bbox indices for all stations
        # (avoids recomputing argmin inside the loop)
        station_bbox_idx = {}
        for x, y, station_id in cal_stations:
            station_id = int(station_id)
            entry = station_masks[station_id]
            if entry is None:
                continue
            _, r_min, r_max, c_min, c_max = entry
            bbox_lon = lon_array[[c_min, c_max]]
            bbox_lat = lat_array[[r_min, r_max]]
            ci_min = np.argmin(np.abs(clim_lon - bbox_lon[0]))
            ci_max = np.argmin(np.abs(clim_lon - bbox_lon[1]))
            ri_min = np.argmin(np.abs(clim_lat - bbox_lat[0]))
            ri_max = np.argmin(np.abs(clim_lat - bbox_lat[1]))
            station_bbox_idx[station_id] = (ci_min, ci_max, ri_min, ri_max, bbox_lon, bbox_lat)

        # now loop stations — all numpy, no disk I/O
        for x, y, station_id in cal_stations:
            station_id = int(station_id)
            entry = station_masks[station_id]
            if entry is None:
                continue

            mask_cut, r_min, r_max, c_min, c_max = entry
            ci_min, ci_max, ri_min, ri_max, bbox_lon, bbox_lat = station_bbox_idx[station_id]

            # pure numpy slice — microseconds
            da_cut = arr[:, ri_min:ri_max+1, ci_min:ci_max+1]  # (366, H, W)

            if da_cut.shape[1:] != mask_cut.shape:
                continue

            # spatial aggregation -> catchment timeseries
            pixels = da_cut[:, mask_cut]  # (366, n_pixels)

            area_cut = get_area_cut(area_array, area_lon, area_lat, bbox_lon, bbox_lat)
            if area_cut.shape == mask_cut.shape:
                w     = area_cut[mask_cut].astype(float)
                w     = np.where(np.isfinite(w), w, 0.0)
                w_sum = w.sum()
                ts    = np.sum(pixels * w[np.newaxis, :], axis=1) / w_sum
            else:
                ts = np.nanmean(pixels, axis=1)

            # annual value
            if var == "ta":
                annual_val = float(np.nanmean(ts))
            else:
                annual_val = float(np.nansum(ts))
            results_accum[station_id][var]["annual"].append(annual_val)

            # monthly climatology
            for m in range(1, 13):
                idx = months == m
                if idx.sum() == 0:
                    continue
                if var == "ta":
                    monthly_val = float(np.nanmean(ts[idx]))
                else:
                    monthly_val = float(np.nansum(ts[idx]))
                results_accum[station_id][var]["monthly_sums"][m-1]   += monthly_val
                results_accum[station_id][var]["monthly_counts"][m-1] += 1

        # free memory before next year
        del arr


# =============================================================================
# STEP 2 — aggregate years -> one row per station
# =============================================================================

results = []
for _, _, station_id in cal_stations:
    station_id = int(station_id)
    row = {"ID": station_id}

    tp_annual_mean  = np.nan
    et0_annual_mean = np.nan

    for var in CLIM_VARS:
        acc    = results_accum[station_id][var]
        annual = [v for v in acc["annual"] if np.isfinite(v)]

        monthly_clim = np.where(
            acc["monthly_counts"] > 0,
            acc["monthly_sums"] / acc["monthly_counts"],
            np.nan
        )

        if var == "ta":
            row["ta_mean"]         = float(np.mean(annual))        if annual else np.nan
            row["ta_std_interann"] = float(np.std(annual, ddof=1)) if len(annual) > 1 else np.nan
            row["ta_seasonality"]  = seasonality_index(monthly_clim)
        else:
            row[f"{var}_mean_annual"]  = float(np.mean(annual))        if annual else np.nan
            row[f"{var}_std_interann"] = float(np.std(annual, ddof=1)) if len(annual) > 1 else np.nan
            row[f"{var}_seasonality"]  = seasonality_index(monthly_clim)

            if var == "tp":
                tp_annual_mean  = row["tp_mean_annual"]
            if var == "eT0":
                et0_annual_mean = row["eT0_mean_annual"]

    if np.isfinite(tp_annual_mean) and tp_annual_mean > 0:
        row["aridity_index"] = float(et0_annual_mean / tp_annual_mean)
    else:
        row["aridity_index"] = np.nan

    results.append(row)


# =============================================================================
# STEP 3 — merge and save
# =============================================================================

df_clim = pd.DataFrame(results)
df_clim["ID"] = df_clim["ID"].astype(int)
glofas5_base_info["ID"] = glofas5_base_info["ID"].astype(int)

glofas5_enriched_clim = glofas5_base_info.merge(df_clim, on="ID", how="left")

print(f"Final table shape: {glofas5_enriched_clim.shape}")
print(glofas5_enriched_clim.head())

# glofas5_enriched_clim.to_csv("glofas5_catchment_meteo_attributes.csv", index=False)
# glofas5_enriched_clim.to_parquet("glofas5_catchment_meteo_attributes.parquet", index=False)
print("\nDone.")

Loading LDD...
LDD shape: (3000, 7200)
Loading pixel area...
Computing BFS catchment masks for all stations...


BFS masks: 100%|██████████| 5379/5379 [07:17<00:00, 12.30it/s]


All masks done in 437.5s


tp:   0%|          | 0/30 [00:00<?, ?it/s]

  Loading glofas_tp_1991.nc into memory... 63.1 GB in 97.4s


...

  Loading glofas_ta_2019.nc into memory... 63.1 GB in 76.2s


ta:  97%|█████████▋| 29/30 [47:16<01:38, 98.41s/it]

  Loading glofas_ta_2020.nc into memory... 63.2 GB in 76.8s


ta: 100%|██████████| 30/30 [48:53<00:00, 97.80s/it]


Final table shape: (5379, 56)
   ID                      name basin        river  provider iso  \
0   4                     Barby  Elbe         Elbe      1004  DE   
1   5  Wittenberg / Lutherstadt  Elbe         Elbe      1004  DE   
2   6                Malliss OP  Elbe  Muritz-Elde      1004  DE   
3   9               Rathenow UP  Elbe  Lower Havel      1004  DE   
4  11            Calbe Grizehne  Elbe        Saale      1004  DE   

   DrainageArea_prov  DrainageArea_LDD        lat       long  ...  \
0           94060.00          93699.47  51.984836  11.882244  ...   
1           61879.00          60938.53  51.856531  12.646309  ...   
2            2879.00           3161.95  53.190627  11.345254  ...   
3           19288.46          18271.30  52.607446  12.321014  ...   
4           23719.00          23560.46  51.916414  11.812211  ...   

   tp_mean_annual  tp_std_interann tp_seasonality eT0_mean_annual  \
0      749.970862        95.141664       0.176231      658.274305   
1      7

In [25]:
# save the results
# glofas5_enriched_clim.to_csv("GloFASv5_Stations_Overview_260223/GloFASv5_catchments_clim_comprehensive.csv", index=False)
glofas5_enriched_clim.to_csv("GloFASv5_catchments_clim_comprehensive.csv", index=False)
